# 365 Probabilidades - Dia #154
## Qual a probabilidade de a regra dos cinco segundos te salvar?

**Tipo:** Curiosidade · Cotidiano
**Data de publicação:** 2026-11-14
**Ferramenta:** Python
**Decisão analisada:** Vale pegar do chão o que acabou de cair?
**Hashtag:** #365Probabilidades #Dia154

---

### 📖 A História

Passei uns dias com os meus sobrinhos recentemente. Bastou um dia para eu me lembrar de algo que fiz muito com o meu filho. E a noite foi para procurar dados sobre isso.

Todo mundo já fez. A comida cai, a mão vai rápido, alguém conta até cinco e a dúvida acaba
ali mesmo. Se foi rápido, está limpo.

A regra dos cinco segundos é uma das poucas regras que quase todo mundo conhece e quase
ninguém sabe de onde veio. A pergunta deste dia é se as bactérias respeitam o relógio.

---

### 📚 O Conceito: o que passa do chão para a comida

Quando um alimento encosta numa superfície, parte das bactérias que estavam ali passa para
ele. Os microbiologistas chamam isso de transferência e medem em porcentagem: de todas as
bactérias que estavam no pedaço de chão, quantas foram parar na comida.

A regra dos cinco segundos supõe que essa passagem precisa de tempo. Se a comida sair do
chão rápido, não daria tempo de nada passar. Para testar, é preciso controlar o tempo de
contato e variar o resto, o tipo de comida e o tipo de chão.

---

### 🧮 O Modelo

**Fontes:**
- Miranda, R. C. & Schaffner, D. W. (2016). Longer contact times increase cross-contamination
  of *Enterobacter aerogenes* from surfaces to food. *Applied and Environmental Microbiology*,
  82(21), 6490-6496. Universidade Rutgers. Bactéria sem risco que gruda nas superfícies como a
  salmonela. 4 alimentos (melancia, pão, pão com manteiga, bala de goma) × 4 superfícies (inox,
  cerâmica, madeira, carpete) × 4 tempos (menos de 1, 5, 30 e 300 segundos) × 2 meios de
  preparo da bactéria. **128 cenários com 20 repetições, 2.560 medições.** Tabelas S1 a S4 do
  material suplementar (média, desvio-padrão, mediana, mínimo, máximo e repetições abaixo do
  limite de detecção de cada cenário), baixadas por Ana em 05/11/2026 e organizadas em
  `dados154/dia-154-transferencia-128-cenarios.csv`.
- Dawson, P., Han, I., Cox, M., Black, C. & Simmons, L. (2007). Residence time and food contact
  time effects on transfer of *Salmonella* Typhimurium from tile, wood and carpet: testing the
  five-second rule. *Journal of Applied Microbiology*, 102(4), 945-953. Universidade Clemson.
  Salmonela sobrevivendo até 4 semanas em superfície seca; em 5 segundos, mais de 99% passou da
  cerâmica para mortadela; menos de 0,5% do carpete, de 5% a 68% da madeira e da cerâmica.
  Números conferidos pelo resumo, fora do modelo.
- Universidade Aston (2014), comunicado sobre uma pesquisa com estudantes: 87% comem ou
  comeriam algo que caiu no chão. Sem tamanho de amostra e sem revisão por pares. Só como
  contexto, fora do modelo.

**Desenho e família do teste.** O experimento é fatorial e balanceado, com 20 repetições por
cenário. A partir das médias e dos desvios-padrão publicados, o notebook:
- refaz a decomposição da variação (soma de quadrados) para dizer quanto de toda a diferença
  entre as 2.560 medições vem do alimento, da superfície, do tempo e do meio;
- compara menos de 1 segundo com 5 segundos em cada um dos 32 cenários por teste t de Welch,
  com correção de Bonferroni para 32 comparações;
- simula 200 mil quedas com alimento e chão sorteados, tirando cada medição de uma normal
  com a média e o desvio de seu cenário, na escala do logaritmo da porcentagem.

"Te salvar" é definido como passar menos de 1% das bactérias do chão para a comida.

**Sem ×0,80.** São medições de laboratório.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 154
rng = np.random.default_rng(SEMENTE)
SORTEIOS = 200_000
PASTA = 'dados154'

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
CORES_ALIM = {'melancia': VERMELHO, 'pão': DOURADO, 'pão com manteiga': '#8a6d3b', 'bala de goma': VERDE}
RODAPE = ('Fonte: Miranda & Schaffner 2016, Applied and Environmental Microbiology, tabelas S1-S4 '
          '· cálculo próprio | #365Probabilidades')

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(log_pct):
    return 10 ** log_pct

print("✅ Bibliotecas carregadas · semente", SEMENTE)


In [ ]:
# --- DADOS ---
ARQ = os.path.join(PASTA, 'dia-154-transferencia-128-cenarios.csv')
if not os.path.exists(ARQ):
    # refaz a tabela a partir da planilha suplementar original
    bruto = pd.read_excel(os.path.join(PASTA, 'miranda-schaffner-2016-suplemento.xlsx'), header=None)
    alimentos = {0: 'melancia', 36: 'pão', 72: 'pão com manteiga', 108: 'bala de goma'}
    pisos = {'Stainless Steel': 'inox', 'Tile': 'cerâmica', 'Wood': 'madeira', 'Carpet': 'carpete'}
    linhas, a, s, t = [], None, None, None
    for i, r in bruto.iterrows():
        if i in alimentos:
            a = alimentos[i]
            continue
        v = r.values
        if isinstance(v[0], str) and v[0].strip() in pisos:
            s = pisos[v[0].strip()]
        if pd.notna(v[1]) and str(v[1]).strip() not in ('Contact', 'time (s)'):
            t = str(v[1]).strip()
        if v[2] in ('TSB', 'Buffer'):
            k, n = str(v[9]).split('/')
            linhas.append(dict(alimento=a, piso=s, tempo_s=t, meio=v[2], media_log=float(v[3]),
                               dp_log=float(v[4]), mediana_log=float(v[5]), min_log=float(v[6]),
                               max_log=float(v[7]), abaixo_deteccao=int(k), n=int(n)))
    pd.DataFrame(linhas).to_csv(ARQ, index=False)

d = pd.read_csv(ARQ)
TEMPOS = ['<1', '5', '30', '300']
ALIMENTOS = ['melancia', 'pão', 'pão com manteiga', 'bala de goma']
PISOS = ['inox', 'cerâmica', 'madeira', 'carpete']
N_REP = 20

print("=" * 72)
print("  DADOS · MIRANDA & SCHAFFNER 2016")
print("=" * 72)
print(f"  Cenários: {len(d)} · repetições por cenário: {N_REP} · medições: {len(d) * N_REP:,}".replace(",", "."))
print(f"  Medições abaixo do limite de detecção: {d['abaixo_deteccao'].sum()} de {len(d) * N_REP}")
print("  (abaixo do limite, o artigo registra o valor do próprio limite)")
print("\n  Transferência média (média geométrica, %) por alimento e tempo:")
tab = d.pivot_table(index='alimento', columns='tempo_s', values='media_log')[TEMPOS].loc[ALIMENTOS]
print(pct(tab).round(1).to_string())
print("=" * 72)


In [ ]:
# --- O MODELO ---

# CAMADA 1 · A tabela dos cinco segundos
print("=" * 72)
print("  CAMADA 1 · O QUE PASSA EM 5 SEGUNDOS (% das bactérias do chão, média geométrica)")
print("=" * 72)
T5 = d[d['tempo_s'] == '5'].pivot_table(index='alimento', columns='piso', values='media_log')
T5 = T5.loc[ALIMENTOS, PISOS]
print(pct(T5).round(1).to_string())
ACIMA10 = int((T5 > 1).sum().sum())
print(f"\n  Combinações de alimento e chão com mais de 10% em 5 segundos: {ACIMA10} de 16")

# CAMADA 2 · Quanto cada fator pesa
print("\n" + "=" * 72)
print("  CAMADA 2 · QUANTO CADA COISA PESA (decomposição da variação das 2.560 medições)")
print("=" * 72)
g = d['media_log'].mean()
ss_dentro = ((N_REP - 1) * d['dp_log'] ** 2).sum()
ss_entre = (N_REP * (d['media_log'] - g) ** 2).sum()
ss_total = ss_dentro + ss_entre
PESO = {}
for fator, nome in [('alimento', 'Alimento'), ('tempo_s', 'Tempo no chão'), ('piso', 'Tipo de chão'),
                    ('meio', 'Meio de preparo')]:
    m = d.groupby(fator)['media_log'].mean()
    ss = (N_REP * len(d) / len(m) * (m - g) ** 2).sum()
    PESO[nome] = ss / ss_total
PESO['Combinações entre eles'] = (ss_entre / ss_total) - sum(PESO.values())
PESO['Repetição a repetição'] = ss_dentro / ss_total
for k, v in PESO.items():
    print(f"  {k:<26} {vir(v * 100):>5}%")

# CAMADA 3 · Menos de 1 segundo contra 5 segundos, cenário a cenário
print("\n" + "=" * 72)
print("  CAMADA 3 · MENOS DE 1 SEGUNDO CONTRA 5 SEGUNDOS (32 cenários, Welch + Bonferroni)")
print("=" * 72)
chave = ['alimento', 'piso', 'meio']
a1 = d[d['tempo_s'] == '<1'].set_index(chave).sort_index()
a5 = d[d['tempo_s'] == '5'].set_index(chave).sort_index()
t_, p_ = stats.ttest_ind_from_stats(a5['media_log'], a5['dp_log'].clip(lower=0.005), N_REP,
                                    a1['media_log'], a1['dp_log'].clip(lower=0.005), N_REP, equal_var=False)
COMP = pd.DataFrame({'dif_log': a5['media_log'] - a1['media_log'], 'p': p_}, index=a1.index)
ALFA = 0.05 / len(COMP)
COMP['mais_em_5s'] = (COMP['p'] < ALFA) & (COMP['dif_log'] > 0)
MAIS5 = int(COMP['mais_em_5s'].sum())
print(f"  Cenários em que 5 segundos passam mais bactéria que menos de 1: {MAIS5} de {len(COMP)}")
print(COMP.groupby(level='alimento')['mais_em_5s'].sum().loc[ALIMENTOS].to_string())

# CAMADA 4 · 200 mil quedas
print("\n" + "=" * 72)
print("  CAMADA 4 · 200 MIL QUEDAS (alimento e chão sorteados)")
print("=" * 72)
def quedas(sub, sorteios=SORTEIOS):
    sub = sub.reset_index(drop=True)
    i = rng.integers(0, len(sub), sorteios)
    return rng.normal(sub['media_log'].values[i], sub['dp_log'].values[i])

QUEDA = {}
for t in TEMPOS:
    x = quedas(d[d['tempo_s'] == t])
    QUEDA[t] = {'salva': (x < 0).mean(), 'mais10': (x > 1).mean()}
    print(f"  {t:>3} s · salva (menos de 1%): {vir(QUEDA[t]['salva'] * 100)}%"
          f" · passa mais de 10%: {vir(QUEDA[t]['mais10'] * 100)}%")

print("\n  Só os cinco segundos, por alimento:")
QUEDA5 = {}
for alim in ALIMENTOS:
    x = quedas(d[(d['tempo_s'] == '5') & (d['alimento'] == alim)])
    QUEDA5[alim] = (x < 0).mean()
    print(f"  {alim:<18} salva em {vir(QUEDA5[alim] * 100)}% das quedas")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · A tabela dos cinco segundos
fig1, ax1 = plt.subplots(figsize=(11, 7))
v = T5.values
im = ax1.imshow(v, cmap='YlOrRd', vmin=-1.2, vmax=2, aspect='auto')
for i in range(v.shape[0]):
    for j in range(v.shape[1]):
        p = pct(v[i, j])
        txt = "≤0,1%" if p < 0.15 else (f"{vir(p, 1)}%" if p < 10 else f"{p:.0f}%")
        ax1.text(j, i, txt, ha='center', va='center', fontsize=15, fontweight='bold',
                 color='white' if v[i, j] > 1 else '#222')
ax1.set_xticks(range(len(PISOS)))
ax1.set_xticklabels(PISOS, fontsize=12)
ax1.set_yticks(range(len(ALIMENTOS)))
ax1.set_yticklabels(ALIMENTOS, fontsize=12)
ax1.grid(False)
ax1.set_title('Quanto das bactérias do chão passa para a comida em 5 segundos\n'
              'média geométrica de 40 medições por célula', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-154-grafico-01-a-tabela-dos-cinco-segundos.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · O tempo, alimento por alimento
fig2, ax2 = plt.subplots(figsize=(11, 7))
x = np.arange(len(TEMPOS))
for alim in ALIMENTOS:
    y = [pct(d[(d['alimento'] == alim) & (d['tempo_s'] == t)]['media_log'].mean()) for t in TEMPOS]
    ax2.plot(x, y, '-o', color=CORES_ALIM[alim], linewidth=2.5, markersize=9, label=alim)
ax2.set_yscale('log')
ax2.set_yticks([0.1, 1, 10, 100])
ax2.set_yticklabels(['0,1%', '1%', '10%', '100%'])
ax2.axvspan(-0.3, 1.3, color=CINZA, alpha=0.08)
ax2.text(0.5, 70, 'a regra', ha='center', fontsize=11, color=CINZA)
ax2.set_xticks(x)
ax2.set_xticklabels(['menos de 1 s', '5 s', '30 s', '5 min'], fontsize=12)
ax2.set_ylabel('Bactérias que passam para a comida (escala log)', fontsize=11)
ax2.legend(frameon=False, fontsize=11)
ax2.set_title('Na melancia, o relógio não muda nada. No pão, cinco segundos já fazem diferença\n'
              'média geométrica sobre os quatro tipos de chão', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-154-grafico-02-o-relogio.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Quanto cada coisa pesa
fig3, ax3 = plt.subplots(figsize=(11, 6.5))
nomes = list(PESO)
vals = [PESO[k] * 100 for k in nomes]
cores = [VERMELHO if k == 'Alimento' else DOURADO if k == 'Tempo no chão' else CINZA for k in nomes]
y = np.arange(len(nomes))[::-1]
ax3.barh(y, vals, color=cores, height=0.6)
for yi, val in zip(y, vals):
    ax3.text(val + 0.6, yi, f"{vir(val)}%", va='center', fontsize=12)
ax3.set_yticks(y)
ax3.set_yticklabels(nomes, fontsize=12)
ax3.set_xlim(0, max(vals) + 8)
ax3.set_xlabel('Parte da variação total das 2.560 medições (%)', fontsize=11)
ax3.set_title('A assinatura estatística: o que caiu pesa quase três vezes mais que o tempo\n'
              'decomposição da soma de quadrados refeita a partir das tabelas S1 a S4', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-154-grafico-03-o-que-pesa.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · 200 mil quedas
fig4, ax4 = plt.subplots(figsize=(11, 6.5))
sal = [QUEDA[t]['salva'] * 100 for t in TEMPOS]
ax4.bar(x, sal, color=[VERDE, DOURADO, CINZA, CINZA], width=0.55)
for xi, val in zip(x, sal):
    ax4.text(xi, val + 1.5, f"{round(val)} em 100", ha='center', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(['menos de 1 s', '5 s', '30 s', '5 min'], fontsize=12)
ax4.set_ylim(0, 100)
ax4.set_ylabel('Quedas em que passa menos de 1% das bactérias', fontsize=11)
ax4.set_title('Em 100 quedas com 5 segundos no chão, a regra salva cerca de metade\n'
              'Monte Carlo com ' + f'{SORTEIOS:,}'.replace(",", ".") + ' quedas, alimento e chão sorteados',
              fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-154-grafico-04-cem-quedas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Com cinco segundos no chão, a regra salva a comida em cerca de 48 de 100 quedas. Pegando na
hora, em cerca de 72. E o que mais decide não é o relógio, é o que caiu. Em menos de um
segundo, uma melancia já pega 9 de cada 10 bactérias de um piso de inox. Em cinco segundos,
uma bala de goma pega 1 de cada mil ou menos.**

A regra não está totalmente errada. No pão, cinco segundos já passam muito mais bactéria do
que um segundo. Mas o tempo explica uma parte pequena da história. O tipo de comida pesa quase
três vezes mais, e o carpete, que parece o pior chão, é o que menos passa bactéria.

Isso fala de transferência, não de doença. Um chão de casa raramente tem tanta bactéria quanto
o chão do laboratório.

*Qual foi a última coisa que você pegou do chão, e ela era mais parecida com a melancia ou com a
bala de goma?*

---

### ⚠️ Limitações do Modelo

- O experimento usou uma bactéria sem risco, em quantidade muito maior do que um chão comum
  costuma ter. Mede quanto passa, não a chance de alguém ficar doente.
- Abaixo do limite de detecção, o artigo registra o valor do próprio limite. Em cenários com
  muitas repetições assim (bala de goma, carpete, pão em menos de 1 segundo), a transferência
  real pode ser ainda menor.
- A simulação tira cada medição de uma normal com a média e o desvio de seu cenário. Isso é uma
  aproximação, porque os valores no limite de detecção não seguem uma normal.
- O sorteio das quedas dá o mesmo peso aos quatro alimentos, aos quatro tipos de chão e aos dois
  meios de preparo. Na vida real, cada pessoa deixa cair coisas diferentes em chãos diferentes.
- O estudo de Dawson et al. 2007 entra só pelos números do resumo.
- A pesquisa da Universidade Aston não informa quantas pessoas responderam e não passou por
  revisão. Fica fora do modelo.
- Sem ×0,80, porque são medições de laboratório.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
